In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as colors

import numpy as np

import cartopy.crs as ccrs

import torch
import torch.nn as nn
import torch.utils.data as data
import torch_geometric
from torch.nn import Sequential as Seq, Linear, ReLU
from matplotlib.animation import FuncAnimation
import torch.distributed as dist

import sys
sys.path.append("../src/")
from utils.data_utils import *
from utils.climate_utils import *
from utils.subgrid_utils import *

from torch.utils.data import Dataset, DataLoader
import os 
import sys
import random

/ext3/miniconda3/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Init
args = {}
save_data = True
exp_num_in = "3"
exp_num_extra = "12"
exp_num_out = "2"
region =  "Gulf_Stream_Ext"  
network = "Transformer"
N_samples = 4000 #4000
N_val = 300 #300
N_test = 500 #500
factor = 10
interval = 1 #2
hist = 0
lag = 1
lam = 50
steps = 8
Nb = 4
lateral = True
load = False
save_model = True
epochs = 8
batch_size = 6
step_weights = [[1],[.2,.8],[.2,.2,.6],[0,0,.2,.8],[0,0,.1,.2,.7],[0,0,0,.1,.2,.7],[0,0,0,0,.1,.2,.7]]
step_lrs = [1e-4,5e-5,1e-5,1e-5,5e-6,5e-6,5e-6]
rand_seed = 1
device = "cuda"
inpt_dict = {"1":["um","vm"],"2":["um","vm","ur","vr"],"3":["um","vm","Tm"],
            "4":["um","vm","ur","vr","Tm","Tr"],"5":["ur","vr"],"6":["ur","vr","Tr"],
            "7":["Tm"],"8":["Tm","Tr"],"9":["u","v"],"10":["u","v","T"],
            "11":["tau_u","tau_v"],"12":["tau_u","tau_v","t_ref"]} 
extra_dict = {"1":["ur","vr"],"2":["ur","vr","Tm"],
            "3":["Tm"],"4":["ur","vr","Tm","Tr"],"5":[],"6":["um","vm"],
             "7":["um","vm","Tm"], "8": ["um","vm","Tm","Tr"],
              "9":["ur","vr","tau_u","tau_v"],"10":["tau_u","tau_v"],
              "11":["t_ref"],"12":["tau_u","tau_v","t_ref"],
             "13":["ur","vr","Tr","tau_u","tau_v","t_ref"]} 
out_dict = {"1":["um","vm"],"2":["um","vm","Tm"],"3":["ur","vr"],
           "4":["ur","vr","Tr"],"5":["u","v"],"6":["u","v","T"]}

# Set seed
torch.manual_seed(rand_seed)
random.seed(rand_seed)
np.random.seed(rand_seed)

# Env variables
os.environ['MASTER_ADDR'] = 'localhost' 
os.environ['MASTER_PORT'] = '1324'

# if region == "Quiescent":
#     interval = 1
if N_samples > 2000:
    interval = 1 
    
s_train = lag*hist # 1*0=0
e_train = s_train + N_samples*interval # 0 + 4000*1 = 4000
e_test = e_train + interval*N_val # 4000 + 1*300 = 4300

In [3]:
# Str video init
if len(sys.argv) > 4:
    n_cond = int((len(sys.argv)-4)/2)

str_video = ""

try:
    for i in range(n_cond):
        if type(globals()[sys.argv[int(4 + i*2)]]) == str:
            temp = str(sys.argv[int(5 + i*2)])
            exec(sys.argv[int(4 + i*2)] +"= temp" )
            if sys.argv[int(4 + i*2)] == "network":
                continue            
            str_video += "_" + sys.argv[int(4 + i*2)] + "_" + sys.argv[int(5 + i*2)]
        elif type(globals()[sys.argv[int(4 + i*2)]]) == int:
            exec(sys.argv[int(4 + i*2)] +"=" + "int(" + sys.argv[int(5 + i*2)] +")" )
            str_video += "_" + sys.argv[int(4 + i*2)] + "_" + sys.argv[int(5 + i*2)]
    print(str_video)
except:
    print("no cond")

str_video += "_Lateral_Data_025_no_smooth"

no cond


In [4]:
# Region init
if region == "Kuroshio":
    lat = [15,41]
    lon = [-215, -185]
elif region == "Kuroshio_Ext":
    lat = [5,50]
    lon = [-250, -175]      
elif region == "Gulf_Stream":
    lat = [25, 50]
    lon = [-70,-35]
elif region == "Gulf_Stream_Ext":
    lat = [27, 50]
    lon = [-82,-35]       
elif region == "Tropics":
    lat = [-5,25]
    lon = [-95,-65]  
elif region == "Tropics_Ext":
    lat = [-5,25]
    lon = [-115,-45]     
elif region == "South_America":
    lat = [-60, -30]
    lon = [-70,-35] 
elif region == "Africa":
    lat = [-50, -20]
    lon = [5,45] 
elif region == "Africa_Ext":
    lat = [-55, -15]
    lon = [-5,55]    
elif region == "Quiescent":
    lat = [-42.5, -17.5]
    lon = [-155,-120] 
elif region == "Quiescent_Ext":
    lat = [-55, -10]
    lon = [-170,-110]            
elif region == "Pacific":
    lat = [-35, 35]
    lon = [-230,-80]     
elif region == "Indian":
    lat = [-30, 28]
    lon = [30,79]   


In [5]:
# Getting input, extra input and output
inputs = inpt_dict[exp_num_in]
extra_in = extra_dict[exp_num_extra]
outputs = out_dict[exp_num_out]

str_in = "".join([i + "_" for i in inputs])
str_ext = "".join([i + "_" for i in extra_in])
str_out = "".join([i + "_" for i in outputs])

print("inputs: " + str_in)
print("extra inputs: " + str_ext)
print("outputs: " + str_out)

N_atm = len(extra_in)
N_in = len(inputs)
N_extra = N_atm + N_in
N_out = len(outputs)

num_in = int((hist+1)*N_in + N_extra)
print("Number of inputs: ", num_in) # 3 (ocean speeds + ocean temp)(t) +
                                    # 3 (atm wind stresses + atm temp)(t) +
                                    # 3 (boundary ocean speeds + boundary ocean temp)(t) -> 3 (ocean speeds + ocean temp)(t+1)
print("Number of outputs: ", N_out)

# Gets required input + extra input -> output within lat long range
inputs, extra_in, outputs = gen_data_025_lateral(inputs,extra_in,outputs,lag,lat,lon,Nb)

inputs: um_vm_Tm_
extra inputs: tau_u_tau_v_t_ref_
outputs: um_vm_Tm_
Number of inputs:  9
Number of outputs:  3


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Data_Functions.py:535: UserWarning: rename 'lat' to 'yu_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yu_ocean","lon":"xu_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Data_Functions.py:535: UserWarning: rename 'lon' to 'xu_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yu_ocean","lon":"xu_ocean"})


In [6]:
# Area selection
grids = xr.open_dataset('/scratch/as15415/Data/CM2x_grids/Grid_cm25_Vertices.nc')
if region == "global_25":
    grids = xr.open_dataset('/scratch/as15415/Data/CM2x_grids/Grid_cm25_Vertices.nc')

elif "global" in region:
    grids = coarse_grid(grids,factor)

else:
    grids = grids.sel({"yu_ocean":slice(lat[0],lat[1]),"xu_ocean":slice(lon[0],lon[1])})


area = torch.from_numpy(grids["area_C"].to_numpy()).to(device=device)
print(area.shape)

torch.Size([119, 189])


In [7]:
# Wet 
wet = xr.zeros_like(inputs[0][0])
# inputs[0][0,12,12] = np.nan
for data in inputs:
    wet +=np.isnan(data[0])
wet = np.isnan(xr.where(wet==0,np.nan,0))
wet = np.nan_to_num(wet.to_numpy())
wet = torch.from_numpy(wet).type(torch.float32).to(device="cpu") # change to device
print(wet.shape)

torch.Size([119, 189])


In [8]:
# Validation set
data_in_val = gen_data_in(0,e_train,e_test,interval,lag,hist,inputs,extra_in)
data_out_val = gen_data_out(0,e_train,e_test,lag,interval,outputs)

val_data = data_CNN_Lateral(data_in_val,data_out_val,wet,N_atm,Nb,device)

print(data_in_val.shape)
print(data_out_val.shape)

(300, 119, 189, 9)
(300, 119, 189, 3)


In [9]:
# Defining args
args["save_data"] = save_data
args["region"] = region
args["network"] = network
args["interval"] = interval
args["N_samples"] = N_samples
args["N_val"] = N_val
args["N_test"] = N_test
args["factor"] = factor
args["hist"] = hist
args["lag"] = lag
args["steps"] = steps
args["str_video"] = str_video
args["s_train"] = s_train
args["e_train"] = e_train
args["e_test"] = e_test
args["inputs"] = inputs
args["extra_in"] = extra_in
args["outputs"] = outputs
args["wet"] = wet
args["area"] = area
args["N_extra"] = N_extra
args["N_in"] = N_in
args["N_out"] = N_out
args["N_atm"] = N_atm
args["num_in"] = num_in
args["str_in"] = str_in
args["str_ext"] = str_ext 
args["str_out"] = str_out
args["Nb"] = Nb
args["lateral"] = lateral
args ["load"] = load
args["val_data"] = val_data
args["save_model"] = save_model
args["World_Size"] = int(torch.cuda.device_count())
args["epochs"] = epochs
args["batch_size"] = batch_size
args["lam"] = lam/1000
args["step_weights"] = step_weights
args["step_lrs"] = step_lrs

In [12]:
data_in_train = []
data_out_train = []
for i in range(args["steps"]):
    offset = 0*args["interval"]
    data_in_train.append(gen_data_in(i,args["s_train"]+offset,args["e_train"],
                                     args["interval"]*args["World_Size"],args["lag"],
                                     args["hist"],args["inputs"],args["extra_in"]))
    data_out_train.append(gen_data_out(i,args["s_train"]+offset,args["e_train"],
                                       args["lag"],args["interval"]*args["World_Size"],
                                       args["outputs"]))

In [13]:
train_data = data_CNN_steps_Lateral(data_in_train,data_out_train,
                                        args["steps"],args["wet"],args["N_atm"],args["Nb"],device=device)  

val_data = args["val_data"]
    
train_sampler = torch.utils.data.distributed.DistributedSampler(
train_data,
num_replicas=1,
rank=0
)

test_sampler = torch.utils.data.distributed.DistributedSampler(
val_data,
num_replicas=args["World_Size"],
rank=0
)

train_loader = torch_geometric.loader.DataLoader(train_data, batch_size=args["batch_size"], sampler=train_sampler)
test_loader = torch_geometric.loader.DataLoader(val_data, batch_size=args["batch_size"], sampler=test_sampler)

In [15]:
if args["save_data"]:
    torch.save(train_data, '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/data/train_data_{0}.pt'.format('steps_'+str(args["steps"])+"_"+args["region"]+'_Test_in_' + args["str_in"] + 'ext_' + args["str_ext"] +'_out'+args['str_out']+'N_train_' + str(args["N_samples"]) + args["str_video"]))
    torch.save(val_data, '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/data/val_data_{0}.pt'.format('steps_'+str(args["steps"])+"_"+args["region"]+'_Test_in_' + args["str_in"] + 'ext_' + args["str_ext"] +'_out'+args['str_out']+'N_train_' + str(args["N_samples"]) + args["str_video"]))